# **LSTM RNN (WORD PREDICTION/SENTENCE COMPLETION)**
## **Project Overview:**

This project aims to develop a deep learn model for prediciting the next word in a given sequence of words. The model is built using Long-Short Term Memory(LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

1. **Data Collection:** We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.
2. **Data Preprocesing:** The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into and testing sets.
3. **Model Building:** An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.
4. **Model Training:** The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.
5. **Model Evaluation:** The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.
6. **Deployment:** A streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time



In [36]:
## Data Collection
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg

import pandas as pd
import numpy as np
import string
import re

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [37]:
## load the dataset
data=gutenberg.raw('shakespeare-hamlet.txt')
## save to a file
with open('hamlet.txt','w',encoding='utf-8') as f:
  f.write(data)

In [38]:
## Data Preprocessing

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,LSTM,Embedding, Dropout,Input
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [39]:
## load  the dataset
with open('hamlet.txt','r',encoding='utf-8') as f:
  text=f.read().lower()

In [40]:
## Tokenize the text
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])
total_words=len(tokenizer.word_index)+1
total_words

4818

In [41]:
## lets view the word index:
tokenizer.word_index


{'the': 1,
 'and': 2,
 'to': 3,
 'of': 4,
 'i': 5,
 'you': 6,
 'a': 7,
 'my': 8,
 'it': 9,
 'in': 10,
 'that': 11,
 'ham': 12,
 'is': 13,
 'not': 14,
 'his': 15,
 'this': 16,
 'with': 17,
 'your': 18,
 'but': 19,
 'for': 20,
 'me': 21,
 'lord': 22,
 'as': 23,
 'what': 24,
 'he': 25,
 'be': 26,
 'so': 27,
 'him': 28,
 'haue': 29,
 'king': 30,
 'will': 31,
 'no': 32,
 'our': 33,
 'we': 34,
 'on': 35,
 'are': 36,
 'if': 37,
 'all': 38,
 'then': 39,
 'shall': 40,
 'by': 41,
 'thou': 42,
 'come': 43,
 'or': 44,
 'hamlet': 45,
 'good': 46,
 'do': 47,
 'hor': 48,
 'her': 49,
 'let': 50,
 'now': 51,
 'thy': 52,
 'how': 53,
 'more': 54,
 'they': 55,
 'from': 56,
 'enter': 57,
 'at': 58,
 'was': 59,
 'oh': 60,
 'like': 61,
 'most': 62,
 'there': 63,
 'well': 64,
 'know': 65,
 'selfe': 66,
 'would': 67,
 'them': 68,
 'loue': 69,
 'may': 70,
 "'tis": 71,
 'vs': 72,
 'sir': 73,
 'qu': 74,
 'which': 75,
 'did': 76,
 'why': 77,
 'laer': 78,
 'giue': 79,
 'thee': 80,
 'ile': 81,
 'must': 82,
 'hath': 

The tokenizer generally breaks the sentences in to words and assign indexes to them.

In [42]:
## create input sequences

inputsequences=[]

for line in text.split('\n'):
  token_list=tokenizer.texts_to_sequences([line])[0]
  for i in range(1,len(token_list)):
    n_gram_sequence=token_list[:i+1]
    inputsequences.append(n_gram_sequence)

inputsequences[:10]

## This will split the text to sentences and convert them to list of indexes but iterating to produce the n_grams

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1180, 1889],
 [1180, 1889, 1890],
 [1180, 1889, 1890, 1891]]

In [43]:
len(inputsequences)

25732

In [44]:
## Pad sequences
max_sequence_len=max([len(x) for x in inputsequences])
inputsequences=np.array(pad_sequences(inputsequences,maxlen=max_sequence_len,padding='pre'))
inputsequences

## THis will iterate through each sentence and pad zeroes to ensure they are of equal length

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]], dtype=int32)

In [45]:
## the length of a sentence
len(inputsequences[0])

14

In [46]:
## SPlit to training and testing

### create predictors and label
import tensorflow as tf
x=inputsequences[:,:-1] ## The X will be all the words in a sentence except the last word
y=inputsequences[:,-1] ## the y will take all the last words of each sentence


In [47]:
## To convert the y indexes to categories (because some will repeat obviously)
y=to_categorical(y,num_classes=total_words)

In [48]:
## Split to train and test
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

### Train the LSTM RNN

In [49]:
# ## define model
# model=Sequential()
# model.add(Embedding(total_words,100,input_length=max_sequence_len)) ## Embedding layer
# model.add(LSTM(150,return_sequences=True))
# model.add(Dropout(0.2)) ## to avoid overfitting
# model.add(LSTM(100))
# model.add(Dense(total_words,activation='softmax')) ## output layer uses softmax because it is multiclassification output
# print(model.summary())

In [50]:
## Alternative way
# Define model
model = Sequential([
    Input(shape=(max_sequence_len - 1,)), ## To make sure we match with the lenth of the X variable
    Embedding(input_dim=total_words, output_dim=100),
    LSTM(150, return_sequences=True),
    Dropout(0.2),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

In [51]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [52]:
model.compile(
    loss='categorical_crossentropy', ## because it is multiclass
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 13, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [54]:
## Train the model
history=model.fit(
    x_train,y_train,
    epochs=100,
    validation_data=(x_test,y_test),verbose=1,
    # callbacks=[early_stop]
  )

Epoch 1/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.0412 - loss: 6.4333 - val_accuracy: 0.0410 - val_loss: 6.9039
Epoch 2/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.0490 - loss: 6.2296 - val_accuracy: 0.0488 - val_loss: 6.9534
Epoch 3/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.0515 - loss: 6.0887 - val_accuracy: 0.0525 - val_loss: 7.0215
Epoch 4/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.0541 - loss: 5.9710 - val_accuracy: 0.0515 - val_loss: 7.0455
Epoch 5/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.0576 - loss: 5.8573 - val_accuracy: 0.0554 - val_loss: 7.1083
Epoch 6/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.0613 - loss: 5.7348 - val_accuracy: 0.0563 - val_loss: 7.1779
Epoch 7/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.0672 - loss: 5.6106 - val_accuracy: 0.0670 - val_loss: 7.2366
Epoch 8/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.0769 - loss: 5.4834 - 

The validation accuracy is poor. But we will move on to predictions in another notebook. We will revisit this another time.

In [55]:
model.save('lstm_model.keras')

In [57]:
## download to local machine

from google.colab import files
files.download('lstm_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>